# FonoKids -- Extraccion de landmarks (video) y embeddings (audio) en Colab

Este notebook hace EXACTAMENTE lo mismo que `scripts/extraer_landmarks_npy.py` y la extraccion de embeddings de `scripts/audio_pronunciation.py` de tu PC, pero corriendo en la GPU gratuita de Colab y leyendo/escribiendo desde tu Google Drive -- no entrena ningun modelo, solo saca los datos procesados que despues usarias para entrenar.

- **Video**: detecta los landmarks de la boca en cada `.avi` (HRNet/FAN, normalizado contra las COMISURAS DE LA BOCA, no los ojos), calcula posicion + velocidad + aceleracion por frame, y guarda un `.npy` por clip. No descarta ningun clip salvo que NINGUN frame haya detectado cara (mismo criterio que venimos usando: no achicar tus datos).
- **Audio**: pasa cada `.wav` por wav2vec2 (`facebook/wav2vec2-xlsr-53-espeak-cv-ft`) y guarda el embedding (vector de 1024 numeros) de cada clip.

### Antes de correr esto

1. Subi tu carpeta `data/sesiones_continuas/` completa (tal cual la tenes en tu PC, con los `.avi` y `.wav` de cada clase) a tu Google Drive, en `MyDrive/FonoKids/data/sesiones_continuas/`.
2. Activa GPU: **Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU** (T4 alcanza).
3. Corre las celdas de arriba hacia abajo.

Si tu carpeta en Drive tiene otro nombre, cambia `DRIVE_BASE_DIR` en la celda de configuracion.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Instalar dependencias

Colab ya trae `torch` (con CUDA), `numpy`, `opencv-python`. Aca agregamos lo que falta: `face-alignment` (detector HRNet/FAN de landmarks) y `transformers` + `huggingface_hub` + `torchaudio` (wav2vec2 para audio).

In [ ]:
!pip install -q face-alignment transformers huggingface_hub
import torch
print('CUDA disponible:', torch.cuda.is_available())

## 3. Configuracion -- ruta base en Drive

In [ ]:
import os
import re

# RUTA DE ENTRADA: donde subiste data/sesiones_continuas/ (tus .avi y .wav).
DRIVE_BASE_DIR = '/content/drive/MyDrive/FonoKids'


def resolver_carpeta_drive(url_o_ruta, mount_root='/content/drive'):
    """Convierte un LINK de una carpeta de Google Drive (el que copiaste con
    'Compartir > Copiar enlace' o el que ves en la barra de direcciones al
    abrirla en drive.google.com) en una ruta de archivo usable dentro de
    Colab. Tambien acepta una ruta normal tipo '/content/drive/MyDrive/...'
    tal cual, por si preferis pegarla directo.

    Requiere autorizar acceso a Drive (te va a pedir loguearte) la primera
    vez que se usa en la sesion."""
    texto = url_o_ruta.strip()

    match = re.search(r"/folders/([a-zA-Z0-9_-]+)", texto)
    if not match:
        # No es un link de carpeta -- asumimos que ya es una ruta de archivo.
        return texto

    folder_id = match.group(1)

    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    service = build("drive", "v3")

    partes = []
    file_id = folder_id
    while True:
        meta = service.files().get(
            fileId=file_id, fields="id, name, parents", supportsAllDrives=True
        ).execute()
        partes.append(meta["name"])
        parents = meta.get("parents")
        if not parents:
            break
        file_id = parents[0]

    partes = list(reversed(partes))
    if partes and partes[0].strip().lower() in ("my drive", "mi unidad"):
        partes = partes[1:]

    ruta = os.path.join(mount_root, "MyDrive", *partes)
    print(f"Link resuelto a: {ruta}")
    return ruta


# =====================================================================
# RUTA DE SALIDA -- pone ACA el link de la carpeta de Drive donde queres
# que se guarden los .npy (landmarks de video + embeddings de audio).
#
# Como conseguir el link: en drive.google.com, entra (o crea) la carpeta
# donde queres la salida, click derecho > "Compartir" > "Copiar enlace"
# (no hace falta cambiar los permisos de quien puede verlo, alcanza con
# copiar el link) y pegalo abajo entre comillas. Tambien podes pegar una
# ruta normal si ya la sabes (ej. '/content/drive/MyDrive/FonoKids').
#
# Si dejas OUTPUT_DRIVE_LINK vacio (''), usa la misma carpeta de entrada.
# =====================================================================
OUTPUT_DRIVE_LINK = ''  # <-- pega aca tu link de Drive

OUTPUT_BASE_DIR = resolver_carpeta_drive(OUTPUT_DRIVE_LINK) if OUTPUT_DRIVE_LINK else DRIVE_BASE_DIR

RUTA_CLIPS = os.path.join(DRIVE_BASE_DIR, 'data', 'sesiones_continuas')
LANDMARKS_OUT_DIR = os.path.join(OUTPUT_BASE_DIR, 'data', 'landmarks_npy')
LABEL_MAP_PATH = os.path.join(LANDMARKS_OUT_DIR, 'label_map.json')
AUDIO_EMB_OUT_DIR = os.path.join(OUTPUT_BASE_DIR, 'data', 'audio_embeddings_cache')

os.makedirs(LANDMARKS_OUT_DIR, exist_ok=True)
os.makedirs(AUDIO_EMB_OUT_DIR, exist_ok=True)

assert os.path.isdir(RUTA_CLIPS), f'No encontre {RUTA_CLIPS} -- revisa que subiste bien la carpeta a Drive.'
clases = sorted(os.listdir(RUTA_CLIPS))
print(f'Entrada (clips):    {RUTA_CLIPS}')
print(f'Salida (landmarks): {LANDMARKS_OUT_DIR}')
print(f'Salida (audio):     {AUDIO_EMB_OUT_DIR}')
print(f'\nEncontre {len(clases)} clases en Drive:')
for c in clases:
    clips_dir = os.path.join(RUTA_CLIPS, c, 'clips')
    n_avi = len([f for f in os.listdir(clips_dir) if f.lower().endswith('.avi')]) if os.path.isdir(clips_dir) else 0
    n_wav = len([f for f in os.listdir(clips_dir) if f.lower().endswith('.wav')]) if os.path.isdir(clips_dir) else 0
    print(f'  {c}: {n_avi} .avi | {n_wav} .wav')

## 4. VIDEO -- codigo de extraccion de landmarks (identico al de tu PC)

Detector HRNet/FAN, normalizacion contra las comisuras de la boca, relleno bfill/ffill de frames sin cara detectada, suavizado exponencial y calculo de dinamica (velocidad + aceleracion). Un clip solo se descarta si NINGUN frame detecto cara -- todo lo demas se guarda igual, tal como venimos haciendo con tus datos.

In [ ]:
import math
import json
import cv2
import numpy as np

# Esquema iBUG/300W de 68 puntos (HRNet/FAN): boca = indices 48-67.
LIP_INDICES = list(range(48, 68))
MOUTH_RIGHT_CORNER = 48
MOUTH_LEFT_CORNER = 54

POSITION_DIM = len(LIP_INDICES) * 2
LANDMARK_DIM = POSITION_DIM * 3
SCALE_EPSILON = 1e-6
OUTLIER_MAX_JUMP = 0.15
SMOOTHING_ALPHA = 0.4
HRNET_BATCH_CHUNK_SIZE = 4

# Umbral informativo (no se descarta nada por esto, solo se avisa en el log) --
# pedido explicito: no achicar el dataset por baja tasa de deteccion.
MIN_DETECTION_RATE_TRAIN = 0.6


def build_hrnet_detector():
    import face_alignment
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cpu':
        print('[AVISO] No hay GPU -- activa GPU en Colab, si no esto va a ser muy lento.')
    detector = face_alignment.FaceAlignment(
        face_alignment.LandmarksType.TWO_D, device=device, flip_input=False, compile=False,
    )
    print(f'>>> Detector HRNet/FAN corriendo en {device.upper()}. <<<')
    return detector


def detect_hrnet(landmarker, frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    preds = landmarker.get_landmarks(rgb)
    if not preds:
        return None
    return preds[0].astype(np.float32)


def normalize_lip_landmarks(landmarks_px):
    """Normaliza contra las COMISURAS DE LA BOCA (no los ojos) -- asi el
    encuadre de los ojos no importa, solo que la boca se vea."""
    right_corner = landmarks_px[MOUTH_RIGHT_CORNER]
    left_corner = landmarks_px[MOUTH_LEFT_CORNER]
    mouth_center = (right_corner + left_corner) / 2.0
    mouth_vector = left_corner - right_corner
    scale = np.linalg.norm(mouth_vector)
    if scale < 1e-3:
        return None
    angle = math.atan2(mouth_vector[1], mouth_vector[0])
    cos_a, sin_a = math.cos(-angle), math.sin(-angle)
    rotation = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float64)
    lip_pts = landmarks_px[LIP_INDICES].astype(np.float64)
    translated = lip_pts - mouth_center
    rotated = translated @ rotation.T
    normalized = rotated / (scale + SCALE_EPSILON)
    return normalized.astype(np.float32).flatten()


def is_outlier(vector, last_valid_vector):
    if last_valid_vector is None:
        return False
    return float(np.abs(vector - last_valid_vector).max()) > OUTLIER_MAX_JUMP


def fill_missing_frames(positions):
    primeros_validos = [i for i, p in enumerate(positions) if p is not None]
    if not primeros_validos:
        return None
    primer_valido = primeros_validos[0]
    ultimo_valido_idx = primeros_validos[-1]
    relleno = list(positions)
    for i in range(primer_valido):
        relleno[i] = relleno[primer_valido]
    i = primer_valido
    while i < ultimo_valido_idx:
        if relleno[i + 1] is None:
            j = i + 1
            while relleno[j] is None:
                j += 1
            inicio, fin = relleno[i], relleno[j]
            pasos = j - i
            for k in range(1, pasos):
                t = k / pasos
                relleno[i + k] = (1 - t) * inicio + t * fin
            i = j
        else:
            i += 1
    for i in range(ultimo_valido_idx + 1, len(relleno)):
        relleno[i] = relleno[ultimo_valido_idx]
    return np.stack(relleno, axis=0).astype(np.float32)


def smooth_positions(positions, alpha=SMOOTHING_ALPHA):
    smoothed = np.empty_like(positions)
    smoothed[0] = positions[0]
    for t in range(1, positions.shape[0]):
        smoothed[t] = alpha * positions[t] + (1 - alpha) * smoothed[t - 1]
    return smoothed.astype(np.float32)


def add_dynamics(positions):
    velocity = np.zeros_like(positions)
    velocity[1:] = positions[1:] - positions[:-1]
    acceleration = np.zeros_like(positions)
    acceleration[1:] = velocity[1:] - velocity[:-1]
    return np.concatenate([positions, velocity, acceleration], axis=1).astype(np.float32)


def process_clip(landmarker, video_path):
    cap = cv2.VideoCapture(video_path)
    positions = []
    detected, total = 0, 0
    last_valid_vector = None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        total += 1
        landmarks_px = detect_hrnet(landmarker, frame)
        vector = normalize_lip_landmarks(landmarks_px) if landmarks_px is not None else None
        if vector is not None and is_outlier(vector, last_valid_vector):
            vector = None
        if vector is not None:
            detected += 1
            last_valid_vector = vector
        positions.append(vector)
    cap.release()
    filled = fill_missing_frames(positions)
    if filled is None:
        return None, detected, total
    smoothed = smooth_positions(filled)
    sequence = add_dynamics(smoothed)
    return sequence, detected, total


print(f'Dimension de salida por frame: {LANDMARK_DIM} (posicion {POSITION_DIM} + velocidad {POSITION_DIM} + aceleracion {POSITION_DIM})')

## 5. VIDEO -- correr la extraccion

Salta automaticamente los clips que ya tengan `.npy` de una corrida anterior -- podes cortar esta celda y volver a correrla sin perder lo ya procesado.

In [ ]:
landmarker = build_hrnet_detector()

label_map = {}
if os.path.exists(LABEL_MAP_PATH):
    with open(LABEL_MAP_PATH, 'r', encoding='utf-8') as f:
        label_map = json.load(f)

total_ok, total_fail = 0, 0
for word in clases:
    clips_dir = os.path.join(RUTA_CLIPS, word, 'clips')
    if not os.path.isdir(clips_dir):
        continue
    clip_paths = sorted(os.path.join(clips_dir, f) for f in os.listdir(clips_dir) if f.lower().endswith('.avi'))
    if not clip_paths:
        print(f'[{word}] sin clips .avi, se omite.')
        continue

    if word not in label_map:
        label_map[word] = len(label_map)
        os.makedirs(LANDMARKS_OUT_DIR, exist_ok=True)
        with open(LABEL_MAP_PATH, 'w', encoding='utf-8') as f:
            json.dump(label_map, f, ensure_ascii=False, indent=2)

    out_dir = os.path.join(LANDMARKS_OUT_DIR, word)
    os.makedirs(out_dir, exist_ok=True)

    pendientes = [cp for cp in clip_paths
                  if not os.path.exists(os.path.join(out_dir, os.path.splitext(os.path.basename(cp))[0] + '.npy'))]
    ya_hechos = len(clip_paths) - len(pendientes)
    print(f'\n[{word}] {ya_hechos} ya tenian landmarks (se saltean), {len(pendientes)} nuevos a procesar.')

    for i, clip_path in enumerate(pendientes, 1):
        sequence, detected, total = process_clip(landmarker, clip_path)
        if sequence is None:
            print(f'  [{word} {i}/{len(pendientes)}] {os.path.basename(clip_path)}: NINGUN frame detecto cara -- descartado.')
            total_fail += 1
            continue
        detection_rate = detected / total if total else 0
        if detection_rate < MIN_DETECTION_RATE_TRAIN:
            print(f'  [{word} {i}/{len(pendientes)}] {os.path.basename(clip_path)}: cara en {detected}/{total} frames ({detection_rate*100:.0f}%) -- se guarda igual.')
        base_name = os.path.splitext(os.path.basename(clip_path))[0]
        np.save(os.path.join(out_dir, f'{base_name}.npy'), sequence)
        total_ok += 1
        if i % 25 == 0 or i == len(pendientes):
            print(f'  [{word} {i}/{len(pendientes)}] {base_name}: cara en {detected}/{total} frames -> guardado {sequence.shape}')

print(f'\n=== VIDEO TOTAL: {total_ok} guardados | {total_fail} descartados (sin cara en ningun frame) ===')
print(f'Landmarks en: {LANDMARKS_OUT_DIR}')

## 6. AUDIO -- cargar wav2vec2

`facebook/wav2vec2-xlsr-53-espeak-cv-ft`, el mismo modelo que usa `scripts/audio_pronunciation.py` en tu PC. Se carga desde su config + pesos a mano (evita el bloqueo de `transformers` al cargar `pytorch_model.bin` con torch < 2.6, ver CVE-2025-32434 -- se usa `weights_only=True`, seguro igual).

In [ ]:
from huggingface_hub import hf_hub_download
from transformers import Wav2Vec2Config, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC
import torchaudio

MODEL_NAME = 'facebook/wav2vec2-xlsr-53-espeak-cv-ft'
TARGET_SAMPLE_RATE = 16000
EMBEDDING_DIM = 1024

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cargando {MODEL_NAME} (primera vez tarda, baja ~1.2GB)...')

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
config = Wav2Vec2Config.from_pretrained(MODEL_NAME)
audio_model = Wav2Vec2ForCTC(config)
weights_path = hf_hub_download(MODEL_NAME, 'pytorch_model.bin')
state_dict = torch.load(weights_path, map_location='cpu', weights_only=True)
audio_model.load_state_dict(state_dict)
audio_model.to(device)
audio_model.eval()
print(f'Listo. Device: {device}.')


def load_audio_16k(wav_path):
    waveform, sr = torchaudio.load(wav_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != TARGET_SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, TARGET_SAMPLE_RATE)
    return waveform.squeeze(0)


def extract_embedding(wav_path):
    """Vector fijo (1024,) que resume el audio completo del clip -- promedio
    en el tiempo del ultimo estado oculto del encoder wav2vec2 (antes de la
    cabeza CTC de fonemas)."""
    waveform = load_audio_16k(wav_path)
    inputs = feature_extractor(waveform.numpy(), sampling_rate=TARGET_SAMPLE_RATE, return_tensors='pt')
    with torch.inference_mode():
        hidden = audio_model.wav2vec2(inputs.input_values.to(device)).last_hidden_state
        pooled = hidden.mean(dim=1).squeeze(0)
    return pooled.cpu().numpy()

## 7. AUDIO -- correr la extraccion de embeddings

Guarda un `.npy` (1024 numeros) por cada `.wav`, con el mismo nombre del clip -- igual criterio que `train_audio_classifier.py` en tu PC (cachea en `data/audio_embeddings_cache/` para no recalcular). Salta los que ya esten calculados de una corrida anterior.

In [ ]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

os.makedirs(AUDIO_EMB_OUT_DIR, exist_ok=True)

clips_por_clase = {}
for word in clases:
    clips_dir = os.path.join(RUTA_CLIPS, word, 'clips')
    if not os.path.isdir(clips_dir):
        continue
    wavs = sorted(os.path.join(clips_dir, f) for f in os.listdir(clips_dir) if f.lower().endswith('.wav'))
    if wavs:
        clips_por_clase[word] = wavs

total = sum(len(v) for v in clips_por_clase.values())
nombres_clases = list(clips_por_clase.keys())
hechos_por_clase = {c: 0 for c in nombres_clases}
nuevos_por_clase = {c: 0 for c in nombres_clases}

REFRESCO_GRAFICO_CADA = 20  # redibuja el grafico de barras cada N clips (no en cada uno, para no frenar la extraccion)

ok, skipped = 0, 0
inicio = time.time()

barra = tqdm(total=total, desc='Embeddings de audio', unit='clip')
for word, wavs in clips_por_clase.items():
    for wav_path in wavs:
        cache_name = os.path.splitext(os.path.basename(wav_path))[0] + '.npy'
        cache_path = os.path.join(AUDIO_EMB_OUT_DIR, cache_name)
        if os.path.exists(cache_path):
            skipped += 1
        else:
            emb = extract_embedding(wav_path)
            np.save(cache_path, emb)
            ok += 1
            nuevos_por_clase[word] += 1
        hechos_por_clase[word] += 1
        barra.update(1)

        if (ok + skipped) % REFRESCO_GRAFICO_CADA == 0 or (ok + skipped) == total:
            clear_output(wait=True)
            transcurrido = time.time() - inicio
            velocidad = (ok + skipped) / transcurrido if transcurrido > 0 else 0
            restante = total - (ok + skipped)
            eta_seg = restante / velocidad if velocidad > 0 else 0

            fig, ax = plt.subplots(figsize=(10, max(3, len(nombres_clases) * 0.35)))
            y_pos = range(len(nombres_clases))
            totales = [len(clips_por_clase[c]) for c in nombres_clases]
            hechos = [hechos_por_clase[c] for c in nombres_clases]
            ax.barh(y_pos, totales, color='#dddddd', label='Total del clip')
            ax.barh(y_pos, hechos, color='#4C72B0', label='Procesados')
            ax.set_yticks(y_pos)
            ax.set_yticklabels(nombres_clases, fontsize=8)
            ax.invert_yaxis()
            ax.set_xlabel('Clips')
            ax.set_title(f'Progreso extraccion de audio -- {ok+skipped}/{total} '
                          f'({100*(ok+skipped)/total:.1f}%) -- ETA {eta_seg/60:.1f} min')
            ax.legend(loc='lower right')
            plt.tight_layout()
            plt.show()
            barra.display()

barra.close()
print(f'\n=== AUDIO TOTAL: {ok} embeddings nuevos | {skipped} ya existian ===')
print(f'Embeddings en: {AUDIO_EMB_OUT_DIR}')

# =====================================================================
# GRAFICO FINAL -- se dibuja UNA vez, al terminar todo, para confirmar
# de un vistazo que quedo completo (todas las barras llenas y verdes).
# =====================================================================
clear_output(wait=True)
fig, ax = plt.subplots(figsize=(10, max(3, len(nombres_clases) * 0.35)))
y_pos = range(len(nombres_clases))
totales = [len(clips_por_clase[c]) for c in nombres_clases]
hechos = [hechos_por_clase[c] for c in nombres_clases]
colores = ['#2ca02c' if h >= t else '#d62728' for h, t in zip(hechos, totales)]
ax.barh(y_pos, hechos, color=colores)
for i, (h, t) in enumerate(zip(hechos, totales)):
    ax.text(h + max(totales) * 0.01, i, f'{h}/{t}', va='center', fontsize=8)
ax.set_yticks(y_pos)
ax.set_yticklabels(nombres_clases, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Clips con embedding')
segundos_totales = time.time() - inicio
ax.set_title(f'EXTRACCION DE AUDIO TERMINADA -- {ok+skipped}/{total} clips '
             f'({segundos_totales/60:.1f} min) -- verde = completa, rojo = incompleta')
plt.tight_layout()
plt.show()

## Listo

Ya quedaron en tu Drive:

- `MyDrive/FonoKids/data/landmarks_npy/<clase>/*.npy` -- landmarks de video, listos para entrenar el modelo visual.
- `MyDrive/FonoKids/data/audio_embeddings_cache/*.npy` -- embeddings de audio, listos para entrenar el clasificador de audio.

Este notebook no entrena ningun modelo -- si despues queres entrenar con estos datos, avisame y armamos esa parte por separado.